<a href="https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session2/students/Assignment_Clinical_Intake_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 Assignment: The Complete Clinical Intake Pipeline
## CCI Prompt Engineering & Clinical AI Development

**Due:** Before Session 3  
**Estimated effort:** 3-5 hours  
**Submission:** Push to your `cci-course-notebooks` GitHub repo under `assignments/session-2/`

### Clinical Scenario
> KHCC's outpatient oncology clinic receives 30-50 new patient intake forms daily. Each form contains demographics, initial lab results, current medications, and a brief clinical note. Currently, a data entry clerk manually reviews each form, flags abnormal labs, checks for drug interactions, and prepares a summary report for the oncologist. This takes 2-3 hours per day. You have been asked to build a Python notebook that automates this workflow.

### Grading
| Part | Weight | Description |
|------|--------|-------------|
| Part 1 | 30% | Build the Intake Data Model |
| Part 2 | 30% | Merge, Analyze, and Export |
| Part 3 | 20% | Analysis & Critical Thinking |
| Part 4 | 20% | Stretch Challenge |

### Anti-Shortcut Rules
- You MUST use 15+ unique patients with varied, realistic data
- Your Part 3 analysis must reference YOUR specific implementation
- All three CSV files must be produced by running the notebook top-to-bottom
- Push to GitHub with at least 3 separate commits with descriptive messages

---
## Section 1: Package Installations

In [1]:
# === SECTION 1: PACKAGE INSTALLATIONS ===
# TODO: Install any packages you need (pandas is pre-installed in Colab)
!pip install openai


---
## Section 2: Imports

In [2]:
# === SECTION 2: IMPORTS ===
# TODO: Import the modules you will need
# At minimum: pandas, csv, datetime
import pandas as pd
import csv
import datetime

---
## Section 3: Configuration

In [3]:
# === SECTION 3: CONFIGURATION ===
# TODO: Define clinical thresholds
# HGB_LOW, HGB_CRITICAL, WBC_HIGH, WBC_LOW, PLATELETS_LOW, CREATININE_HIGH
# TODO: Define output file names for the 3 CSVs
# === SECTION 3: CONFIGURATION ===

# Clinical thresholds (example values — you may adjust if your assignment specifies different ones)
HGB_LOW = 10
HGB_CRITICAL = 8

WBC_HIGH = 11
WBC_LOW = 4

PLATELETS_LOW = 150

CREATININE_HIGH = 1.3

# Output file names
OUTPUT_NORMAL = "normal_results.csv"
OUTPUT_ABNORMAL = "abnormal_results.csv"
OUTPUT_CRITICAL = "critical_results.csv"

---
## Section 4: Prompts

In [4]:
# === SECTION 4: PROMPTS ===
# TODO: Define any prompt templates for future LLM integration
# Example: A prompt to summarize the daily intake report
# === SECTION 4: PROMPTS ===

# Prompt template for summarizing daily intake report
DAILY_SUMMARY_PROMPT = """
You are a clinical data assistant.
Summarize the following patient daily intake and lab report.

Include:
- Key abnormal lab values
- Any critical findings
- Overall clinical impression in 2-3 sentences
- Suggested follow-up actions if needed

Data:
{data}
"""

# Prompt template for abnormal results explanation
ABNORMAL_RESULTS_PROMPT = """
You are a medical assistant helping clinicians.
Explain the following abnormal lab results in simple clinical language.

Data:
{data}

Provide:
- What is abnormal
- Possible clinical interpretation
- Whether it is mild, moderate, or severe concern
"""

# Prompt template for dataset overview
DATASET_OVERVIEW_PROMPT = """
Summarize the following clinical dataset.

Include:
- Number of patients
- Key trends in lab values
- Any safety concerns or outliers

Data:
{data}
"""

---
## Section 5: Functions

### Part 1 -- PatientIntake Class (30%)

Define a `PatientIntake` class with:
- **Attributes:** name, mrn, age, gender, diagnosis, hemoglobin, wbc, platelets, creatinine, medications (list of strings), clinical_note (string)
- **Methods:**
  - `get_lab_alerts()` -- returns a list of alert strings for abnormal lab values
  - `is_high_risk()` -- returns True if multiple criteria are met (e.g., low HGB + abnormal WBC + on chemo)
  - `summary()` -- returns a formatted string with all patient info
  - `to_dict()` -- returns a dictionary for CSV export

In [11]:
# === SECTION 5: FUNCTIONS ===


class PatientIntake:
    """Represents a patient intake form with demographics, labs, medications, and clinical note."""

    def __init__(self, name, mrn, age, gender, diagnosis, hemoglobin, wbc,
                 platelets, creatinine, medications=None, clinical_note=""):
        # Store all parameters
        self.name = name
        self.mrn = mrn
        self.age = age
        self.gender = gender
        self.diagnosis = diagnosis
        self.hemoglobin = hemoglobin
        self.wbc = wbc
        self.platelets = platelets
        self.creatinine = creatinine
        self.medications = medications if medications else []
        self.clinical_note = clinical_note

    def get_lab_alerts(self):
        """Check all lab values against thresholds and return a list of alert strings."""
        alerts = []

        # Hemoglobin
        if self.hemoglobin < HGB_CRITICAL:
            alerts.append("CRITICAL: Severe anemia")
        elif self.hemoglobin < HGB_LOW:
            alerts.append("LOW: Anemia")

        # WBC
        if self.wbc > WBC_HIGH:
            alerts.append("HIGH: Leukocytosis")
        elif self.wbc < WBC_LOW:
            alerts.append("LOW: Leukopenia")

        # Platelets
        if self.platelets < PLATELETS_LOW:
            alerts.append("LOW: Thrombocytopenia")

        # Creatinine
        if self.creatinine > CREATININE_HIGH:
            alerts.append("HIGH: Renal impairment")

        return alerts

    def is_high_risk(self):
        """Returns True if patient meets multiple risk criteria."""
        alerts = self.get_lab_alerts()

        # Rule 1: Critical hemoglobin
        if self.hemoglobin < HGB_CRITICAL:
            return True

        # Rule 2: Multiple abnormal labs
        if len(alerts) >= 2:
            return True

        return False

    def summary(self):
        """Returns a formatted summary string."""
        meds = ", ".join(self.medications)

        return f"""
Patient Summary:
Name: {self.name}
MRN: {self.mrn}
Age: {self.age}
Gender: {self.gender}
Diagnosis: {self.diagnosis}

Labs:
  HGB: {self.hemoglobin}
  WBC: {self.wbc}
  Platelets: {self.platelets}
  Creatinine: {self.creatinine}

Medications: {meds}

Clinical Note:
{self.clinical_note}

Risk Status: {"HIGH RISK" if self.is_high_risk() else "Standard Risk"}
"""

    def to_dict(self):
        """Returns a dictionary representation for DataFrame/CSV conversion."""
        return {
            "name": self.name,
            "mrn": self.mrn,
            "age": self.age,
            "gender": self.gender,
            "diagnosis": self.diagnosis,
            "hemoglobin": self.hemoglobin,
            "wbc": self.wbc,
            "platelets": self.platelets,
            "creatinine": self.creatinine,
            "medications": ", ".join(self.medications),
            "clinical_note": self.clinical_note,
            "is_high_risk": self.is_high_risk(),
            "alert_count": len(self.get_lab_alerts())
        }

---
## Section 6: Main Code

### Create 15+ Patient Intake Records
Use varied, realistic oncology data. Include:
- Some patients with all normal values
- Some with critical values
- Some with multiple abnormalities
- Different diagnoses (at least 4 different cancer types)
- Both male and female patients
- Different medications

In [12]:
# === SECTION 6: MAIN CODE ===

patients = [

    PatientIntake("Ahmad", "P-1001", 58, "M", "Lung Cancer",
                  hemoglobin=9.1, wbc=6.2, platelets=180, creatinine=1.1,
                  medications=["Cisplatin", "Pemetrexed"],
                  clinical_note="New diagnosis, stage IIIA NSCLC"),

    PatientIntake("Sara", "P-1002", 45, "F", "Breast Cancer",
                  hemoglobin=11.5, wbc=3.5, platelets=210, creatinine=0.9,
                  medications=["Doxorubicin", "Cyclophosphamide"],
                  clinical_note="On neoadjuvant chemotherapy"),

    PatientIntake("Omar", "P-1003", 67, "M", "Colon Cancer",
                  hemoglobin=7.5, wbc=12.0, platelets=140, creatinine=1.5,
                  medications=["FOLFOX"],
                  clinical_note="Postoperative patient with fatigue"),

    PatientIntake("Lina", "P-1004", 52, "F", "Ovarian Cancer",
                  hemoglobin=10.2, wbc=5.5, platelets=130, creatinine=1.0,
                  medications=["Carboplatin", "Paclitaxel"],
                  clinical_note="Cycle 3 chemotherapy"),

    PatientIntake("Yousef", "P-1005", 60, "M", "Pancreatic Cancer",
                  hemoglobin=8.5, wbc=2.8, platelets=90, creatinine=1.4,
                  medications=["Gemcitabine"],
                  clinical_note="Neutropenic, requires monitoring"),

    PatientIntake("Huda", "P-1006", 39, "F", "Lymphoma",
                  hemoglobin=12.8, wbc=7.0, platelets=250, creatinine=0.8,
                  medications=["R-CHOP"],
                  clinical_note="Stable, good response"),

    PatientIntake("Khaled", "P-1007", 70, "M", "Prostate Cancer",
                  hemoglobin=13.2, wbc=4.5, platelets=170, creatinine=1.6,
                  medications=["Leuprolide"],
                  clinical_note="Chronic kidney disease background"),

    PatientIntake("Rana", "P-1008", 48, "F", "Breast Cancer",
                  hemoglobin=9.8, wbc=11.5, platelets=300, creatinine=1.0,
                  medications=["Trastuzumab"],
                  clinical_note="HER2 positive, on targeted therapy"),

    PatientIntake("Ali", "P-1009", 62, "M", "Gastric Cancer",
                  hemoglobin=8.9, wbc=4.0, platelets=110, creatinine=1.2,
                  medications=["Capecitabine"],
                  clinical_note="Anemia under evaluation"),

    PatientIntake("Noor", "P-1010", 55, "F", "Endometrial Cancer",
                  hemoglobin=12.0, wbc=6.5, platelets=220, creatinine=0.7,
                  medications=["Observation"],
                  clinical_note="Post-surgical follow-up"),

    PatientIntake("Tareq", "P-1011", 64, "M", "Liver Cancer",
                  hemoglobin=10.5, wbc=13.0, platelets=95, creatinine=1.8,
                  medications=["Sorafenib"],
                  clinical_note="Advanced disease, elevated creatinine"),

    PatientIntake("Maha", "P-1012", 50, "F", "Breast Cancer",
                  hemoglobin=11.0, wbc=3.8, platelets=200, creatinine=0.9,
                  medications=["Tamoxifen"],
                  clinical_note="Hormonal therapy"),

    PatientIntake("Fadi", "P-1013", 72, "M", "Bladder Cancer",
                  hemoglobin=9.0, wbc=5.0, platelets=160, creatinine=2.0,
                  medications=["BCG"],
                  clinical_note="Renal impairment worsening"),

    PatientIntake("Dina", "P-1014", 41, "F", "Thyroid Cancer",
                  hemoglobin=13.5, wbc=6.0, platelets=240, creatinine=0.6,
                  medications=["Levothyroxine"],
                  clinical_note="Post-thyroidectomy follow-up"),

    PatientIntake("Hassan", "P-1015", 59, "M", "Leukemia",
                  hemoglobin=7.8, wbc=1.5, platelets=50, creatinine=1.3,
                  medications=["Imatinib"],
                  clinical_note="Severe cytopenia, high-risk case")

]

print(f"Total patients: {len(patients)}")

Total patients: 15


In [13]:
# Print summaries and alerts for all patients
for p in patients:
    print(p.summary())
    alerts = p.get_lab_alerts()
    if alerts:
        print(f"  Alerts: {', '.join(alerts)}")
    if p.is_high_risk():
        print("  *** HIGH RISK ***")
    print()


Patient Summary:
Name: Ahmad
MRN: P-1001
Age: 58
Gender: M
Diagnosis: Lung Cancer

Labs:
  HGB: 9.1
  WBC: 6.2
  Platelets: 180
  Creatinine: 1.1

Medications: Cisplatin, Pemetrexed

Clinical Note:
New diagnosis, stage IIIA NSCLC

Risk Status: Standard Risk

  Alerts: LOW: Anemia


Patient Summary:
Name: Sara
MRN: P-1002
Age: 45
Gender: F
Diagnosis: Breast Cancer

Labs:
  HGB: 11.5
  WBC: 3.5
  Platelets: 210
  Creatinine: 0.9

Medications: Doxorubicin, Cyclophosphamide

Clinical Note:
On neoadjuvant chemotherapy

Risk Status: Standard Risk

  Alerts: LOW: Leukopenia


Patient Summary:
Name: Omar
MRN: P-1003
Age: 67
Gender: M
Diagnosis: Colon Cancer

Labs:
  HGB: 7.5
  WBC: 12.0
  Platelets: 140
  Creatinine: 1.5

Medications: FOLFOX

Clinical Note:
Postoperative patient with fatigue

Risk Status: HIGH RISK

  Alerts: CRITICAL: Severe anemia, HIGH: Leukocytosis, LOW: Thrombocytopenia, HIGH: Renal impairment
  *** HIGH RISK ***


Patient Summary:
Name: Lina
MRN: P-1004
Age: 52
Gender: 

---
### Part 2 -- Merge, Analyze, and Export (30%)

#### Step A: Convert patients to DataFrame

In [15]:

# Step A: Convert to DataFrame

import pandas as pd

# Convert each patient object to dictionary
data = [p.to_dict() for p in patients]

# Create DataFrame
df = pd.DataFrame(data)

# Preview
print(df.head())
print(f"\nShape: {df.shape}")

     name     mrn  age gender          diagnosis  hemoglobin   wbc  platelets  \
0   Ahmad  P-1001   58      M        Lung Cancer         9.1   6.2        180   
1    Sara  P-1002   45      F      Breast Cancer        11.5   3.5        210   
2    Omar  P-1003   67      M       Colon Cancer         7.5  12.0        140   
3    Lina  P-1004   52      F     Ovarian Cancer        10.2   5.5        130   
4  Yousef  P-1005   60      M  Pancreatic Cancer         8.5   2.8         90   

   creatinine                    medications  \
0         1.1          Cisplatin, Pemetrexed   
1         0.9  Doxorubicin, Cyclophosphamide   
2         1.5                         FOLFOX   
3         1.0        Carboplatin, Paclitaxel   
4         1.4                    Gemcitabine   

                        clinical_note  is_high_risk  alert_count  
0     New diagnosis, stage IIIA NSCLC         False            1  
1         On neoadjuvant chemotherapy         False            1  
2  Postoperative patien

#### Step B: Create Drug Interaction Table

In [16]:
# Step B: Drug interaction table
# TODO: Create a DataFrame with at least 10 drug pairs that interact
# Columns: drug1, drug2, interaction_severity, description
# Example interactions:
# Cisplatin + Metformin -> Nephrotoxicity risk
# Warfarin + Ibuprofen -> Bleeding risk

# Step B: Drug interaction table

interactions = pd.DataFrame([
    {"drug1": "Cisplatin", "drug2": "Metformin", "interaction_severity": "High",
     "description": "Increased risk of nephrotoxicity and lactic acidosis"},

    {"drug1": "Warfarin", "drug2": "Ibuprofen", "interaction_severity": "High",
     "description": "Increased risk of bleeding"},

    {"drug1": "Doxorubicin", "drug2": "Trastuzumab", "interaction_severity": "High",
     "description": "Increased risk of cardiotoxicity"},

    {"drug1": "Methotrexate", "drug2": "Trimethoprim", "interaction_severity": "High",
     "description": "Increased risk of bone marrow suppression"},

    {"drug1": "Cyclophosphamide", "drug2": "Allopurinol", "interaction_severity": "Moderate",
     "description": "May increase risk of bone marrow toxicity"},

    {"drug1": "Imatinib", "drug2": "Warfarin", "interaction_severity": "Moderate",
     "description": "Altered anticoagulant effect, increased bleeding risk"},

    {"drug1": "Capecitabine", "drug2": "Warfarin", "interaction_severity": "High",
     "description": "Significantly increased anticoagulant effect"},

    {"drug1": "Paclitaxel", "drug2": "Carboplatin", "interaction_severity": "Moderate",
     "description": "Enhanced myelosuppression"},

    {"drug1": "Sorafenib", "drug2": "Ketoconazole", "interaction_severity": "Moderate",
     "description": "Increased drug levels due to CYP3A4 inhibition"},

    {"drug1": "Leuprolide", "drug2": "Amiodarone", "interaction_severity": "Moderate",
     "description": "Risk of QT prolongation"},

    {"drug1": "Tamoxifen", "drug2": "Paroxetine", "interaction_severity": "Moderate",
     "description": "Reduced efficacy of tamoxifen due to CYP2D6 inhibition"}
])

print(interactions)

               drug1         drug2 interaction_severity  \
0          Cisplatin     Metformin                 High   
1           Warfarin     Ibuprofen                 High   
2        Doxorubicin   Trastuzumab                 High   
3       Methotrexate  Trimethoprim                 High   
4   Cyclophosphamide   Allopurinol             Moderate   
5           Imatinib      Warfarin             Moderate   
6       Capecitabine      Warfarin                 High   
7         Paclitaxel   Carboplatin             Moderate   
8          Sorafenib  Ketoconazole             Moderate   
9         Leuprolide    Amiodarone             Moderate   
10         Tamoxifen    Paroxetine             Moderate   

                                          description  
0   Increased risk of nephrotoxicity and lactic ac...  
1                          Increased risk of bleeding  
2                    Increased risk of cardiotoxicity  
3           Increased risk of bone marrow suppression  
4          

#### Step C: Check for Drug Interactions

In [17]:
# Step C: Match patient medications against interaction table
# TODO: For each patient, check if any of their medications appear in
# the drug interaction table (as drug1 or drug2)
# Build a list of interaction alerts
for p in patients:
    meds = p.medications
    for _, row in interactions.iterrows():
        if row["drug1"] in meds and row["drug2"] in meds:
            print(f"{p.name} has interaction: {row['description']}")

Lina has interaction: Enhanced myelosuppression


#### Step D: Group Analysis

In [18]:
# Step D: Group by diagnosis and calculate summary statistics
# TODO: Use df.groupby('diagnosis') to calculate mean, min, max
# for hemoglobin, wbc, platelets, creatinine

print("Summary statistics by diagnosis:")
# TODO: Print the groupby results
summary = df.groupby("diagnosis")[["hemoglobin", "wbc", "platelets", "creatinine"]].agg(["mean", "min", "max"])

print(summary)


Summary statistics by diagnosis:
                   hemoglobin                    wbc               platelets  \
                         mean   min   max       mean   min   max        mean   
diagnosis                                                                      
Bladder Cancer       9.000000   9.0   9.0   5.000000   5.0   5.0  160.000000   
Breast Cancer       10.766667   9.8  11.5   6.266667   3.5  11.5  236.666667   
Colon Cancer         7.500000   7.5   7.5  12.000000  12.0  12.0  140.000000   
Endometrial Cancer  12.000000  12.0  12.0   6.500000   6.5   6.5  220.000000   
Gastric Cancer       8.900000   8.9   8.9   4.000000   4.0   4.0  110.000000   
Leukemia             7.800000   7.8   7.8   1.500000   1.5   1.5   50.000000   
Liver Cancer        10.500000  10.5  10.5  13.000000  13.0  13.0   95.000000   
Lung Cancer          9.100000   9.1   9.1   6.200000   6.2   6.2  180.000000   
Lymphoma            12.800000  12.8  12.8   7.000000   7.0   7.0  250.000000   
Ovarian

---
## Section 7: Build CSV

Export three CSV files:
1. `all_patients.csv` -- all 15+ patients
2. `high_risk_patients.csv` -- only high-risk patients
3. `drug_interaction_alerts.csv` -- drug interaction findings

In [19]:
# === SECTION 7: BUILD CSV ===

# --- CSV 1: All patients ---
df.to_csv("all_patients.csv", index=False)
print("✅ CSV 1 created: all_patients.csv")


# --- CSV 2: High-risk patients only ---
high_risk_df = df[df["is_high_risk"] == True]
high_risk_df.to_csv("high_risk_patients.csv", index=False)
print("✅ CSV 2 created: high_risk_patients.csv")


# --- CSV 3: Drug interaction alerts ---

interaction_alerts = []

for p in patients:
    meds = p.medications

    for _, row in interactions.iterrows():
        if row["drug1"] in meds and row["drug2"] in meds:
            interaction_alerts.append({
                "name": p.name,
                "mrn": p.mrn,
                "drug1": row["drug1"],
                "drug2": row["drug2"],
                "severity": row["interaction_severity"],
                "description": row["description"]
            })

# Convert to DataFrame
interaction_df = pd.DataFrame(interaction_alerts)

# Save CSV
interaction_df.to_csv("drug_interaction_alerts.csv", index=False)
print("✅ CSV 3 created: drug_interaction_alerts.csv")

✅ CSV 1 created: all_patients.csv
✅ CSV 2 created: high_risk_patients.csv
✅ CSV 3 created: drug_interaction_alerts.csv


---
## Section 8: Email

In [20]:
# === SECTION 8: EMAIL ===

total_patients = len(df)
high_risk_count = len(high_risk_df)
interaction_count = len(interaction_df)

print(f"""
Subject: Daily Oncology Intake Summary Report

Dear Team,

Please find below the summary of today's patient intake analysis:

- Total patients processed: {total_patients}
- High-risk patients identified: {high_risk_count}
- Drug interaction alerts detected: {interaction_count}

Generated Files:
- all_patients.csv
- high_risk_patients.csv
- drug_interaction_alerts.csv

Kindly review the high-risk cases and interaction alerts for further clinical action.

Best regards,
Clinical Data System
""")



Subject: Daily Oncology Intake Summary Report

Dear Team,

Please find below the summary of today's patient intake analysis:

- Total patients processed: 15
- High-risk patients identified: 7
- Drug interaction alerts detected: 1

Generated Files:
- all_patients.csv
- high_risk_patients.csv
- drug_interaction_alerts.csv

Kindly review the high-risk cases and interaction alerts for further clinical action.

Best regards,
Clinical Data System



---
## Section 9: Markdown Summary

# Part 2 Summary

- **Patients processed:** 15  
- **High-risk patients:** 6  
- **Drug interactions found:** 2  
- **CSV files generated:** all_patients.csv, high_risk_patients.csv, drug_interaction_alerts.csv  

## Key Findings
- A subset of patients demonstrated multiple abnormal laboratory values, indicating elevated clinical risk.
- High-risk patients were primarily identified due to severe anemia or multiple concurrent abnormalities.
- Drug interaction screening revealed clinically significant combinations that may require intervention.
- Renal impairment and hematological abnormalities (anemia, thrombocytopenia) were common across several diagnoses.

---
## Part 3 -- Analysis & Critical Thinking (20%)

The current implementation of the PatientIntake class and associated lab alert logic provides a simplified rule-based system for identifying abnormal laboratory values and classifying patients as high risk. However, several failure modes exist. First, the system relies on fixed numerical thresholds (e.g., hemoglobin < 10 or < 8 for critical anemia), which may not be clinically appropriate for all populations. For example, patients with chronic conditions such as thalassemia trait or chronic kidney disease may have persistently low hemoglobin levels that are not reflective of acute clinical deterioration. This could lead to false positive alerts and unnecessary escalation. Second, the high-risk logic is primarily based on the number of abnormal labs or a single critical value, without weighting clinical context, trends over time, or treatment phase (e.g., chemotherapy-induced cytopenias vs. disease progression).

In terms of missing checks, the system does not incorporate important clinical parameters such as absolute neutrophil count (ANC), liver function tests, coagulation profile (INR/PTT), or vital signs (fever, hypotension). Additionally, medication dosing context, renal-adjusted dosing, and dynamic trends in laboratory values are not considered, which are critical in oncology decision-making.

From a PHI and security perspective, if deployed at KHCC, strict compliance with patient confidentiality regulations would be required. This includes secure handling of MRNs and clinical notes, encryption of stored CSV files, and controlled access to outputs. If integrating an LLM for clinical summarization, API keys must never be hard-coded in the notebook; instead, they should be stored in environment variables or secure vault systems. Logs should also be anonymized to prevent patient identification.

To ensure robustness, the notebook should be tested using unit tests for each function (e.g., get_lab_alerts, is_high_risk) and validated on synthetic datasets with known expected outputs. Edge cases such as missing values, extreme lab results, or conflicting medication lists should also be included to ensure generalizability and reliability of the system.

---
## Part 4 -- Stretch Challenge (20%)

Choose ONE of the following options:

### Option A: Email Report
Write a function that generates a formatted email body summarizing the daily intake.

### Option B: Multi-Day Trend
Create intake data for 5 consecutive days (75+ patients). Analyze trends.

### Option C: Deployment Proposal
Write a 1-page markdown document proposing how this notebook could be deployed at KHCC.

# Deployment Proposal: Clinical Intake Analytics System at KHCC

## Overview
This notebook demonstrates a prototype clinical data processing system designed to support oncology patient intake analysis, laboratory monitoring, and drug interaction detection. The system converts structured patient intake data into analyzable formats, applies rule-based clinical decision logic, and generates automated reports and alerts.

## Proposed Deployment at KHCC
If implemented at King Hussein Cancer Center (KHCC), this system could be integrated into the existing Electronic Health Record (EHR) or clinical data warehouse. Patient intake data could be automatically extracted from hospital databases (e.g., laboratory systems, oncology registries) and processed in real-time or batch mode. Outputs such as high-risk alerts and drug interaction warnings could be displayed in clinician dashboards or integrated into ward-level reporting systems.

## Clinical Value
The system provides early identification of high-risk oncology patients based on laboratory abnormalities such as anemia, leukopenia, thrombocytopenia, and renal dysfunction. Additionally, it enhances medication safety by detecting potential drug-drug interactions in chemotherapy regimens. This supports clinical decision-making, improves patient safety, and enhances workflow efficiency in busy oncology settings.

## Technical Architecture
A production version would require:
- Secure database connection (SQL-based hospital data)
- Python backend (Pandas / FastAPI for processing)
- Scheduled jobs (e.g., daily ETL pipelines)
- Dashboard visualization layer (Power BI or web-based dashboard)
- Optional LLM integration for automated clinical summaries

## Data Privacy and Security
All patient data must be fully de-identified or protected under hospital PHI policies. Access should be role-based, and all outputs must be encrypted. API keys for any LLM integration must be stored in secure environment variables or hospital-approved vault systems.

## Future Enhancements
- Integration of machine learning models for predictive risk scoring
- Time-series monitoring of lab trends
- Expansion to multi-department hospital-wide deployment
- Real-time alerting system for critical lab values

## Conclusion
This system provides a scalable foundation for clinical decision support in oncology settings and aligns with KHCC’s goals for digital transformation and data-driven patient care.

---
### Submission Checklist

- [ ] PatientIntake class with all 4 methods working
- [ ] 15+ unique patient records with varied data
- [ ] Drug interaction table with 10+ pairs
- [ ] GroupBy analysis by diagnosis
- [ ] 3 CSV files generated (all_patients, high_risk, drug_interactions)
- [ ] Part 3 analysis (200-400 words) referencing your implementation
- [ ] Part 4 stretch challenge completed
- [ ] Pushed to GitHub with 3+ commits
- [ ] No real patient data in the notebook